In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
指定ディレクトリ内のすべてのCSVを読み込み、
緯度(lat)・経度(lon)分布について、以下6種類の距離をまとめて計算する

1 Wasserstein distance
2 Jensen-Shannon distance
3 Hellinger distance
4 Total Variation distance
5 Energy distance
6 Sinkhorn distance

出力（各距離ごと）
1 距離行列 CSV
2 距離ヒートマップ PNG
3 隣接ペア時系列 CSV
4 隣接ペア時系列 PNG
5 行総和 PNG

特徴
・ディレクトリを再帰探索
・全CSV対象
・途中経過詳細表示
・各ログに開始時刻 / 現在時刻 / 経過秒 / ジョブID を付与
・Excel互換 cp932 出力
・ヒートマップ軸ラベル自動間引き
・PNG保存に加えて画面表示も行う
・時系列図で外れ値に赤丸を付与
・JS / Hellinger / TV は 2次元ヒストグラム分布で計算
・Wasserstein は lat/lon の1D Wasserstein平均
・Energy は 2次元 energy distance
・Sinkhorn は POT があれば実行
"""

# ========= 設定 =========

DATA_DIR = r"./csv-2026-03-12"
PREFIX = "multi_distance"

MIN_ROWS = 2
OUTPUT_ENCODING = "cp932"
PAIR_PROGRESS_STEP = 10

HEATMAP_MAX_LABELS = 20
HEATMAP_FONT_SIZE = 8

Z_THR = 3.5
OUTLIER_COLOR = "red"

BINS = 60
USE_GLOBAL_RANGE = True
EPS = 1e-12

# Energy / Sinkhorn の計算コスト対策
MAX_POINTS_FOR_POINT_METRICS = 400

# Sinkhorn 正則化パラメータ
SINKHORN_REG = 0.01

# 使用する距離
ENABLE_METRICS = [
    "wasserstein",
    "js",
    "hellinger",
    "tv",
    "energy",
    "sinkhorn",
]

# ========================

import os
import csv
import time
import math
import re
import uuid
import traceback
import numpy as np

from datetime import datetime

from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams

from scipy.spatial.distance import jensenshannon, cdist
from scipy.stats import wasserstein_distance

# Sinkhorn (POT)
try:
    import ot
    HAS_POT = True
except Exception:
    HAS_POT = False


# -----------------------
# 実行情報
# -----------------------

PROGRAM_START = datetime.now()
PROGRAM_START_STR = PROGRAM_START.strftime("%Y-%m-%d %H:%M:%S")
JOB_ID = str(uuid.uuid4())[:8]


# -----------------------
# ログ
# -----------------------

def fmt_sec(sec):
    m = int(sec // 60)
    s = sec % 60
    return f"{m:02d}:{s:05.2f}"


def log(msg):
    now = datetime.now()
    now_str = now.strftime("%Y-%m-%d %H:%M:%S")
    elapsed = (now - PROGRAM_START).total_seconds()

    print(
        f"[JOB {JOB_ID}] "
        f"[START {PROGRAM_START_STR}] "
        f"[NOW {now_str}] "
        f"[ELAPSED {elapsed:8.2f}s] "
        f"{msg}",
        flush=True
    )


def log_section(title):
    log("=" * 100)
    log(title)
    log("=" * 100)


def log_kv(title, **kwargs):
    parts = [title]
    for k, v in kwargs.items():
        parts.append(f"{k}={v}")
    log(" ".join(parts))


# -----------------------
# 日本語フォント
# -----------------------

JP_FONT = [
    "Yu Gothic",
    "Meiryo",
    "IPAexGothic",
    "IPAPGothic",
]

def setup_japanese_font():
    available = {f.name for f in font_manager.fontManager.ttflist}

    for name in JP_FONT:
        if name in available:
            rcParams["font.family"] = "sans-serif"
            rcParams["font.sans-serif"] = [name]
            log(f"日本語フォント設定 {name}")
            break

    rcParams["axes.unicode_minus"] = False


# -----------------------
# CSV読み込み
# -----------------------

READ_ENCODINGS = ("utf-8", "utf-8-sig", "cp932", "iso-8859-1")


def open_read_fallback(path):
    last = None
    for enc in READ_ENCODINGS:
        try:
            return open(path, "r", encoding=enc, newline="")
        except Exception as e:
            last = e

    try:
        return open(path, "r", newline="")
    except Exception:
        raise last or RuntimeError(f"cannot open {path}")


def row_has_header_like(cells):
    if len(cells) < 4:
        return True

    c2 = (cells[2] or "").strip().lower()
    c3 = (cells[3] or "").strip().lower()

    if c2 in ("lat", "latitude") or c3 in ("lon", "lng", "longitude"):
        return True

    try:
        float(cells[2])
        float(cells[3])
        return False
    except Exception:
        return True


def iter_rows_with_optional_header(reader):
    first = next(reader, None)
    if first is None:
        return

    if not row_has_header_like(first):
        yield first

    for row in reader:
        yield row


def load_valid_latlon(filepath):
    lats = []
    lons = []

    total = 0
    valid = 0
    short_rows = 0
    parse_errors = 0
    range_errors = 0

    with open_read_fallback(filepath) as f:
        reader = csv.reader(f)

        for row in iter_rows_with_optional_header(reader):
            total += 1

            if len(row) < 4:
                short_rows += 1
                continue

            try:
                lat = float((row[2] or "").strip())
                lon = float((row[3] or "").strip())

                if math.isfinite(lat) and math.isfinite(lon):
                    if -90 <= lat <= 90 and -180 <= lon <= 180:
                        lats.append(lat)
                        lons.append(lon)
                        valid += 1
                    else:
                        range_errors += 1
                else:
                    parse_errors += 1
            except Exception:
                parse_errors += 1
                continue

    return {
        "lat": np.array(lats, dtype=float),
        "lon": np.array(lons, dtype=float),
        "total": total,
        "valid": valid,
        "short_rows": short_rows,
        "parse_errors": parse_errors,
        "range_errors": range_errors,
    }


# -----------------------
# 補助
# -----------------------

def robust_z(x):
    if len(x) == 0:
        return np.array([], dtype=float)

    med = np.median(x)
    mad = np.median(np.abs(x - med))

    if mad == 0:
        return np.zeros_like(x, dtype=float)

    return 0.6745 * (x - med) / mad


def date_from_filename(f):
    base = os.path.basename(f)

    m1 = re.search(r"(\d{14})", base)
    if m1:
        try:
            return datetime.strptime(m1.group(1), "%Y%m%d%H%M%S")
        except Exception:
            pass

    m2 = re.search(r"(\d{4}-\d{2}-\d{2})[_-](\d{2}-\d{2}-\d{2})", base)
    if m2:
        try:
            return datetime.strptime(
                m2.group(1) + " " + m2.group(2),
                "%Y-%m-%d %H-%M-%S"
            )
        except Exception:
            pass

    key = base.split("-")[0]
    try:
        return datetime.strptime(key[:14], "%Y%m%d%H%M%S")
    except Exception:
        return None


def make_sparse_ticks(labels, max_labels=20):
    n = len(labels)

    if n == 0:
        return [], []

    step = max(1, math.ceil(n / max_labels))
    pos = list(range(0, n, step))

    if pos[-1] != n - 1:
        pos.append(n - 1)

    tick_labels = [labels[i] for i in pos]
    return pos, tick_labels


def make_global_range(records):
    if not USE_GLOBAL_RANGE:
        return (-90.0, 90.0), (-180.0, 180.0)

    all_lat_list = [r["lat"] for r in records if len(r["lat"]) > 0]
    all_lon_list = [r["lon"] for r in records if len(r["lon"]) > 0]

    if not all_lat_list or not all_lon_list:
        return (-90.0, 90.0), (-180.0, 180.0)

    all_lat = np.concatenate(all_lat_list)
    all_lon = np.concatenate(all_lon_list)

    lat_min = float(np.min(all_lat))
    lat_max = float(np.max(all_lat))
    lon_min = float(np.min(all_lon))
    lon_max = float(np.max(all_lon))

    if lat_min == lat_max:
        lat_min -= 0.5
        lat_max += 0.5
    if lon_min == lon_max:
        lon_min -= 0.5
        lon_max += 0.5

    lat_margin = max((lat_max - lat_min) * 0.02, 1e-6)
    lon_margin = max((lon_max - lon_min) * 0.02, 1e-6)

    return (
        (lat_min - lat_margin, lat_max + lat_margin),
        (lon_min - lon_margin, lon_max + lon_margin),
    )


def hist2d_prob(lat, lon, lat_range, lon_range, bins=BINS):
    H, _, _ = np.histogram2d(
        lat, lon,
        bins=[bins, bins],
        range=[lat_range, lon_range]
    )

    P = H.astype(float).ravel()
    s = P.sum()

    if s <= 0:
        return np.full(bins * bins, 1.0 / (bins * bins), dtype=float)

    P /= s
    P = np.maximum(P, EPS)
    P /= P.sum()
    return P


def subsample_points(lat, lon, max_points=400, seed=42):
    n = len(lat)
    pts = np.column_stack([lat, lon])

    if n <= max_points:
        return pts

    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=max_points, replace=False)
    return pts[idx]


# -----------------------
# 距離定義
# -----------------------

def dist_wasserstein_approx(rec1, rec2):
    d_lat = wasserstein_distance(rec1["lat"], rec2["lat"])
    d_lon = wasserstein_distance(rec1["lon"], rec2["lon"])
    return float(0.5 * (d_lat + d_lon))


def dist_js(rec1, rec2):
    return float(jensenshannon(rec1["prob"], rec2["prob"], base=2.0))


def dist_hellinger(rec1, rec2):
    p = rec1["prob"]
    q = rec2["prob"]
    return float(np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)))


def dist_tv(rec1, rec2):
    p = rec1["prob"]
    q = rec2["prob"]
    return float(0.5 * np.sum(np.abs(p - q)))


def mean_pairwise_distance(X, Y):
    D = cdist(X, Y, metric="euclidean")
    return float(np.mean(D))


def dist_energy_2d(rec1, rec2):
    X = rec1["pts_sub"]
    Y = rec2["pts_sub"]

    exy = mean_pairwise_distance(X, Y)
    exx = mean_pairwise_distance(X, X)
    eyy = mean_pairwise_distance(Y, Y)

    val = 2.0 * exy - exx - eyy
    if val < 0 and abs(val) < 1e-12:
        val = 0.0
    return float(np.sqrt(max(val, 0.0)))


def dist_sinkhorn_2d(rec1, rec2, reg=SINKHORN_REG):
    if not HAS_POT:
        return np.nan

    X = rec1["pts_sub"]
    Y = rec2["pts_sub"]

    a = np.full(len(X), 1.0 / len(X), dtype=float)
    b = np.full(len(Y), 1.0 / len(Y), dtype=float)

    M = ot.dist(X, Y, metric="euclidean")
    d = ot.sinkhorn2(a, b, M, reg)
    return float(np.sqrt(max(d, 0.0)))


DISTANCE_FUNCS = {
    "wasserstein": dist_wasserstein_approx,
    "js": dist_js,
    "hellinger": dist_hellinger,
    "tv": dist_tv,
    "energy": dist_energy_2d,
    "sinkhorn": dist_sinkhorn_2d,
}

DISTANCE_LABELS = {
    "wasserstein": "Wasserstein distance",
    "js": "JS distance",
    "hellinger": "Hellinger distance",
    "tv": "Total Variation distance",
    "energy": "Energy distance",
    "sinkhorn": "Sinkhorn distance",
}


# -----------------------
# 出力
# -----------------------

def save_matrix_csv(path, dist, names):
    with open(path, "w", encoding=OUTPUT_ENCODING, newline="", errors="replace") as f:
        w = csv.writer(f)
        w.writerow([""] + names)
        for i in range(len(names)):
            row = []
            for x in dist[i]:
                if np.isfinite(x):
                    row.append(f"{x:.6f}")
                else:
                    row.append("")
            w.writerow([names[i]] + row)


def plot_heatmap(path, dist, names, title, cbar_label):
    tick_positions, tick_labels = make_sparse_ticks(names, HEATMAP_MAX_LABELS)

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(dist, aspect="equal", interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

    ax.set_xticks(tick_positions)
    ax.set_yticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=HEATMAP_FONT_SIZE)
    ax.set_yticklabels(tick_labels, fontsize=HEATMAP_FONT_SIZE)
    ax.set_xlabel("CSV file")
    ax.set_ylabel("CSV file")
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def save_timeseries_csv(path, dates, vals, z, outliers):
    with open(path, "w", encoding=OUTPUT_ENCODING, newline="", errors="replace") as f:
        w = csv.writer(f)
        w.writerow(["date", "distance", "z", "is_outlier"])
        for d, v, zz, out in zip(dates, vals, z, outliers):
            if np.isfinite(v):
                sv = f"{v:.6f}"
                sz = f"{zz:.3f}" if np.isfinite(zz) else ""
                so = int(out)
            else:
                sv = ""
                sz = ""
                so = ""
            w.writerow([d.strftime("%Y-%m-%d %H:%M:%S"), sv, sz, so])


def plot_timeseries(path, dates, vals, z, outliers, title, ylabel):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(dates, vals, "o-", label="distance")
    ax.set_title(title)
    ax.set_xlabel("date")
    ax.set_ylabel(ylabel)

    dates_arr = np.array(dates, dtype=object)
    vals_arr = np.array(vals, dtype=float)
    z_arr = np.array(z, dtype=float)

    finite_mask = np.isfinite(vals_arr)
    out_mask = finite_mask & outliers

    if np.any(out_mask):
        ax.scatter(
            dates_arr[out_mask],
            vals_arr[out_mask],
            s=140,
            facecolors="none",
            edgecolors=OUTLIER_COLOR,
            linewidths=2,
            label=f"outlier |z| >= {Z_THR}"
        )

        for d, v, zz in zip(dates_arr[out_mask], vals_arr[out_mask], z_arr[out_mask]):
            ax.annotate(
                f"{zz:.1f}",
                (d, v),
                xytext=(0, 6),
                textcoords="offset points",
                ha="center"
            )

    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_row_sums(path, dates, row_sums, title, ylabel):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(dates, row_sums, "o-")
    ax.set_title(title)
    ax.set_xlabel("date")
    ax.set_ylabel(ylabel)

    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# -----------------------
# 距離計算本体
# -----------------------

def compute_distance_matrix(records, metric_name, func):
    n = len(records)
    dist = np.zeros((n, n), dtype=float)

    total_pairs = n * (n - 1) // 2
    done = 0
    calc_start = time.time()

    log_kv(
        f"{metric_name} 距離計算開始",
        total_pairs=total_pairs
    )

    for i in range(n):
        for j in range(i + 1, n):
            name_i = os.path.basename(records[i]["path"])
            name_j = os.path.basename(records[j]["path"])

            try:
                d = func(records[i], records[j])
            except Exception as e:
                log(
                    f"{metric_name} 距離計算失敗 "
                    f"i={i} j={j} "
                    f"file1={name_i} file2={name_j} "
                    f"err={repr(e)}"
                )
                d = np.nan

            dist[i, j] = dist[j, i] = d
            done += 1

            if done % PAIR_PROGRESS_STEP == 0 or done == total_pairs:
                elapsed = time.time() - calc_start
                rate = done / elapsed if elapsed > 0 else 0.0
                remain = (total_pairs - done) / rate if rate > 0 else 0.0

                if np.isfinite(d):
                    d_str = f"{d:.6f}"
                else:
                    d_str = "nan"

                log(
                    f"{metric_name} 距離進捗 "
                    f"{done}/{total_pairs} "
                    f"({100.0 * done / total_pairs:.1f}%) "
                    f"current_pair=({i+1},{j+1}) "
                    f"file1={name_i} "
                    f"file2={name_j} "
                    f"last_distance={d_str} "
                    f"elapsed={fmt_sec(elapsed)} "
                    f"remain_est={fmt_sec(remain)}"
                )

    log(f"{metric_name} 距離計算完了 所要時間={fmt_sec(time.time() - calc_start)}")
    return dist


def summarize_changes(metric, dates, vals, z, outliers):
    order = np.argsort(np.nan_to_num(-vals, nan=-np.inf))

    log(f"{metric} 変化上位")
    for k in range(min(10, len(order))):
        idx = order[k]
        if np.isfinite(vals[idx]):
            z_str = f"{z[idx]:.2f}" if np.isfinite(z[idx]) else "nan"
            log(
                f"{metric} rank={k+1} "
                f"date={dates[idx].strftime('%Y-%m-%d %H:%M:%S')} "
                f"dist={vals[idx]:.6f} "
                f"z={z_str}"
            )

    log(f"{metric} 外れ値")
    idxs = np.where(outliers)[0]
    if len(idxs) == 0:
        log(f"{metric} 外れ値なし")
    else:
        for idx in idxs:
            z_str = f"{z[idx]:.2f}" if np.isfinite(z[idx]) else "nan"
            log(
                f"{metric} outlier "
                f"date={dates[idx].strftime('%Y-%m-%d %H:%M:%S')} "
                f"dist={vals[idx]:.6f} "
                f"z={z_str}"
            )


# -----------------------
# メイン
# -----------------------

def main():
    total_start = time.time()

    log_section("処理開始")

    log_kv(
        "実行設定",
        data_dir=os.path.abspath(DATA_DIR),
        prefix=PREFIX,
        min_rows=MIN_ROWS,
        bins=BINS,
        use_global_range=USE_GLOBAL_RANGE,
        z_thr=Z_THR,
        point_cap=MAX_POINTS_FOR_POINT_METRICS,
        sinkhorn_reg=SINKHORN_REG,
    )

    setup_japanese_font()

    if "sinkhorn" in ENABLE_METRICS and not HAS_POT:
        log("POT未導入のため sinkhorn はスキップされます。必要なら pip install pot")

    # -------------------
    # CSV探索
    # -------------------
    scan_start = time.time()
    csv_files = []

    log_section("CSV探索開始")
    for root, _, files in os.walk(DATA_DIR):
        log(f"探索中 root={root}")
        for f in files:
            if f.lower().endswith(".csv"):
                full = os.path.join(root, f)
                csv_files.append(full)
                log(f"CSV発見 path={full}")

    log(f"CSV探索完了 count={len(csv_files)} 所要時間={fmt_sec(time.time() - scan_start)}")

    if len(csv_files) < 2:
        log("CSV不足")
        print("CSV不足")
        return

    # -------------------
    # 読み込み
    # -------------------
    load_start = time.time()
    records = []

    log_section("CSV読み込み開始")

    for i, f in enumerate(sorted(csv_files), 1):
        t0 = time.time()

        dt = date_from_filename(f)
        dt_source = "filename"
        if dt is None:
            dt = datetime.fromtimestamp(os.path.getmtime(f))
            dt_source = "mtime"

        result = load_valid_latlon(f)
        lat = result["lat"]
        lon = result["lon"]

        if len(lat) >= MIN_ROWS:
            records.append({
                "dt": dt,
                "path": f,
                "lat": lat,
                "lon": lon,
            })
            accepted = 1
        else:
            accepted = 0

        log(
            f"CSV読込 "
            f"{i}/{len(csv_files)} "
            f"file={os.path.basename(f)} "
            f"date={dt.strftime('%Y-%m-%d %H:%M:%S')} "
            f"date_source={dt_source} "
            f"rows_total={result['total']} "
            f"valid={result['valid']} "
            f"short_rows={result['short_rows']} "
            f"parse_errors={result['parse_errors']} "
            f"range_errors={result['range_errors']} "
            f"accepted={accepted} "
            f"time={fmt_sec(time.time() - t0)}"
        )

    records.sort(key=lambda x: x["dt"])
    n = len(records)

    log(f"CSV読み込み完了 valid_csv={n} 所要時間={fmt_sec(time.time() - load_start)}")

    if n < 2:
        log("有効CSV不足")
        print("有効CSV不足")
        return

    # -------------------
    # 共通レンジ
    # -------------------
    range_start = time.time()
    lat_range, lon_range = make_global_range(records)
    log(
        f"共通レンジ決定 "
        f"lat_range={lat_range} "
        f"lon_range={lon_range} "
        f"time={fmt_sec(time.time() - range_start)}"
    )

    # -------------------
    # 前処理
    # -------------------
    prep_start = time.time()
    log_section("前処理開始")

    for i, rec in enumerate(records, 1):
        t0 = time.time()

        rec["prob"] = hist2d_prob(rec["lat"], rec["lon"], lat_range, lon_range, bins=BINS)
        rec["pts_sub"] = subsample_points(
            rec["lat"], rec["lon"],
            max_points=MAX_POINTS_FOR_POINT_METRICS,
            seed=42
        )

        log(
            f"前処理 "
            f"{i}/{n} "
            f"file={os.path.basename(rec['path'])} "
            f"raw_points={len(rec['lat'])} "
            f"hist_dim={len(rec['prob'])} "
            f"subsampled_points={len(rec['pts_sub'])} "
            f"time={fmt_sec(time.time() - t0)}"
        )

    log(f"前処理完了 所要時間={fmt_sec(time.time() - prep_start)}")

    names = [os.path.basename(r["path"]) for r in records]
    dates_all = [r["dt"] for r in records]

    active_metrics = []
    for m in ENABLE_METRICS:
        if m == "sinkhorn" and not HAS_POT:
            continue
        active_metrics.append(m)

    log(f"実行対象距離 {', '.join(active_metrics)}")

    # -------------------
    # 各距離
    # -------------------
    for metric in active_metrics:
        metric_start = time.time()
        label = DISTANCE_LABELS[metric]
        func = DISTANCE_FUNCS[metric]

        log_section(f"{metric} 開始 ({label})")

        dist = compute_distance_matrix(records, metric, func)

        matrix_csv = f"{PREFIX}_{metric}_matrix.csv"
        heatmap_png = f"{PREFIX}_{metric}_heatmap.png"
        timeseries_csv = f"{PREFIX}_{metric}_timeseries.csv"
        timeseries_png = f"{PREFIX}_{metric}_timeseries.png"
        row_sums_png = f"{PREFIX}_{metric}_row_sums.png"

        # 距離行列保存
        save_start = time.time()
        log(f"{metric} 保存開始 file={matrix_csv}")
        save_matrix_csv(matrix_csv, dist, names)
        log(f"{metric} 保存完了 file={matrix_csv} time={fmt_sec(time.time() - save_start)}")

        # ヒートマップ
        fig_start = time.time()
        log(f"{metric} 図作成開始 file={heatmap_png}")
        plot_heatmap(
            heatmap_png,
            dist,
            names,
            f"{DISTANCE_LABELS[metric]} ヒートマップ",
            DISTANCE_LABELS[metric]
        )
        log(f"{metric} 図作成完了 file={heatmap_png} time={fmt_sec(time.time() - fig_start)}")

        # 隣接距離
        vals = []
        dates = []

        ts_start = time.time()
        log(f"{metric} 隣接時系列作成開始")

        for i in range(1, len(records)):
            vals.append(dist[i - 1, i])
            dates.append(records[i]["dt"])
            d = dist[i - 1, i]
            d_str = f"{d:.6f}" if np.isfinite(d) else "nan"
            log(
                f"{metric} 隣接距離 "
                f"idx={i}/{len(records)-1} "
                f"prev={os.path.basename(records[i-1]['path'])} "
                f"curr={os.path.basename(records[i]['path'])} "
                f"dist={d_str}"
            )

        vals = np.array(vals, dtype=float)

        finite_mask = np.isfinite(vals)
        z = np.full(len(vals), np.nan, dtype=float)
        outliers = np.zeros(len(vals), dtype=bool)

        if np.any(finite_mask):
            z_finite = robust_z(vals[finite_mask])
            z[finite_mask] = z_finite
            outliers[finite_mask] = np.abs(z_finite) >= Z_THR

        log(f"{metric} 隣接時系列作成完了 time={fmt_sec(time.time() - ts_start)}")

        # 時系列CSV
        save_ts_start = time.time()
        log(f"{metric} 保存開始 file={timeseries_csv}")
        save_timeseries_csv(timeseries_csv, dates, vals, z, outliers)
        log(f"{metric} 保存完了 file={timeseries_csv} time={fmt_sec(time.time() - save_ts_start)}")

        # 時系列図
        fig_ts_start = time.time()
        log(f"{metric} 図作成開始 file={timeseries_png}")
        plot_timeseries(
            timeseries_png,
            dates,
            vals,
            z,
            outliers,
            f"隣接CSV間の {DISTANCE_LABELS[metric]}",
            DISTANCE_LABELS[metric]
        )
        log(f"{metric} 図作成完了 file={timeseries_png} time={fmt_sec(time.time() - fig_ts_start)}")

        # 行総和図
        row_sums = np.nansum(dist, axis=1)

        fig_row_start = time.time()
        log(f"{metric} 図作成開始 file={row_sums_png}")
        plot_row_sums(
            row_sums_png,
            dates_all,
            row_sums,
            f"各CSVの距離行列 行総和 ({DISTANCE_LABELS[metric]})",
            f"row sum of {DISTANCE_LABELS[metric]}"
        )
        log(f"{metric} 図作成完了 file={row_sums_png} time={fmt_sec(time.time() - fig_row_start)}")

        summarize_changes(metric, dates, vals, z, outliers)

        log(f"{metric} 完了 総所要時間={fmt_sec(time.time() - metric_start)}")

    log_section(f"全処理完了 総時間={fmt_sec(time.time() - total_start)}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        log("ユーザー中断 KeyboardInterrupt")
        raise
    except Exception as e:
        log(f"致命的エラー {repr(e)}")
        log(traceback.format_exc())
        raise